# Day 3-03｜ByteTrack-to-BEV 整合輸出
> Python 籃球運動資料分析課程  
> 本單元整合 YOLO detector、ByteTrack、court keypoint Homography 與 BEV 投影，輸出含 track path 的分析影片。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 將 track ID 對應到 player footpoint。
- 使用 court keypoints 估計 Homography 並投影到 BEV。
- 產生 `frame, track_id, bbox, footpoint, bev_x, bev_y` 結構化結果。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 選擇參考影片、detector 與 court keypoint model。
2. 從 court keypoint 較穩定的 frame 開始，執行 ByteTrack 與 BEV 投影。
3. 儲存整合影片、JSON 與 CSV。


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(preferred=COURSE_ROOT_HINT)


In [ ]:
import pandas as pd

from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    court_keypoint_model_path,
    detector_model_path,
    reference_videos,
    write_bytetrack_bev_video,
)

videos = reference_videos(COURSE_ROOT)
if len(videos) < 3:
    raise FileNotFoundError("assets/raw/reference_videos/ 至少需要三支參考影片。")

VIDEO_PATH = videos[2]
DETECTOR_PATH = detector_model_path(COURSE_ROOT)
COURT_MODEL_PATH = court_keypoint_model_path(COURSE_ROOT)
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("start frame:", 30)


In [ ]:
bev_video, records = write_bytetrack_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=COURSE_ROOT / "assets" / "results" / "d3_03_bytetrack_bev.mp4",
    max_frames=90,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=30,
)

tracks = pd.DataFrame(records)
out_csv = COURSE_ROOT / "assets" / "results" / "d3_03_bytetrack_bev.csv"
tracks.to_csv(out_csv, index=False)

print("BEV video:", bev_video)
print("BEV json:", bev_video.with_suffix(".json"))
print("BEV csv:", out_csv)
print("rows:", len(tracks))
display_video_in_notebook(bev_video, loop=True)
tracks.head()


Day 1 到 Day 3 的主流程至此完成：Roboflow 標註格式、YOLO detection、court keypoint Homography、ByteTrack 與 BEV 路徑輸出。輸出流程會檢查 keypoint 數量、RANSAC inliers 與重投影誤差；若當前 frame 的 Homography 不穩定，會沿用前一個穩定矩陣。

## 本單元產出檔案

- `assets/results/d3_03_bytetrack_bev.mp4`：ByteTrack-to-BEV 並排影片。
- `assets/results/d3_03_bytetrack_bev.json`：逐 frame 的追蹤與投影資料。
- `assets/results/d3_03_bytetrack_bev.csv`：可做後續分析的表格化追蹤資料。
